In [ ]:
# ==========================================
# 15 - FEATURE SELECTION EXPERIMENT
# KHỞI TẠO MÔI TRƯỜNG
# ==========================================

import sys
import json
from pathlib import Path

ROOT_DIR = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(ROOT_DIR))

import src.imports as lib

from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    hamming_loss
)

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

RANDOM_SEED = 42
N_SPLITS = 5

lib.np.random.seed(RANDOM_SEED)

id_col = "ma_hoc_sinh"

choice_cols = [
    "Vật lí", "Hóa học", "Sinh học", "Địa lí",
    "Giáo dục kinh tế và pháp luật (GDKT&PL)",
    "Tin học", "Công nghệ", "Âm nhạc", "Mĩ thuật"
]

MODEL_NAMES = [
    "Random Forest",
    "XGBoost",
    "CatBoost"
]

# Đường dẫn
split_dir = ROOT_DIR / "data/data-splits"
fold_dir = split_dir / "cv_folds_v1"

fe_dir = ROOT_DIR / (
    "experiments/v2_optimization/"
    "03_feature_engineering/results/tables"
)

output_dir = ROOT_DIR / (
    "experiments/v2_optimization/"
    "04_feature_selection/results"
)

table_dir = output_dir / "tables"
figure_dir = output_dir / "figures"
config_dir = output_dir / "configs"

for directory in [table_dir, figure_dir, config_dir]:
    directory.mkdir(parents=True, exist_ok=True)

# Đọc Development
df_dev = lib.pd.read_csv(
    split_dir / "development.csv",
    encoding="utf-8-sig"
)

# Đọc đặc trưng đã tạo tại bước 14
X_fe = lib.pd.read_csv(
    fe_dir / "X_development_feature_engineered.csv",
    encoding="utf-8-sig"
)

fe_ids = lib.pd.read_csv(
    fe_dir / "feature_engineering_student_ids.csv",
    encoding="utf-8-sig"
)[id_col]

assert df_dev[id_col].reset_index(drop=True).equals(
    fe_ids.reset_index(drop=True)
), "Thứ tự ID của dữ liệu Feature Engineering không khớp!"

score_cols = [
    col for col in df_dev.columns
    if col.startswith(("L6_", "L7_", "L8_", "L9_"))
    and col.endswith(("_hki", "_hkii", "_ca_nam"))
]

X_original = df_dev[score_cols].copy()
y = df_dev[choice_cols].copy()

assert X_original.shape == (9000, 96)
assert X_fe.shape == (9000, 197)
assert y.shape == (9000, 9)
assert df_dev[id_col].is_unique
assert X_fe.notna().all().all()
assert lib.np.isfinite(X_fe.to_numpy(dtype=float)).all()

lib.pd.testing.assert_frame_equal(
    X_fe[score_cols].reset_index(drop=True),
    X_original.reset_index(drop=True),
    check_dtype=False
)

# Đọc 5 fold cố định
fold_data = []

for fold in range(1, N_SPLITS + 1):

    train_ids = lib.pd.read_csv(
        fold_dir / f"fold_{fold}_train_ids.csv",
        encoding="utf-8-sig"
    )[id_col]

    val_ids = lib.pd.read_csv(
        fold_dir / f"fold_{fold}_validation_ids.csv",
        encoding="utf-8-sig"
    )[id_col]

    assert set(train_ids).isdisjoint(set(val_ids))

    assert (
        set(train_ids) | set(val_ids)
        == set(df_dev[id_col])
    )

    train_idx = lib.np.flatnonzero(
        df_dev[id_col].isin(train_ids).to_numpy()
    )

    val_idx = lib.np.flatnonzero(
        df_dev[id_col].isin(val_ids).to_numpy()
    )

    fold_data.append({
        "fold": fold,
        "train_idx": train_idx,
        "val_idx": val_idx
    })

# Đọc cấu hình mô hình từ V1
config_paths = {
    "Random Forest": ROOT_DIR / (
        "experiments/v1_baseline/"
        "random_forest/results/rf_config.json"
    ),
    "XGBoost": ROOT_DIR / (
        "experiments/v1_baseline/"
        "xgboost/results/xgb_config.json"
    ),
    "CatBoost": ROOT_DIR / (
        "experiments/v1_baseline/"
        "catboost/results/cat_config.json"
    )
}

v1_params = {}

for model_name, path in config_paths.items():

    with open(path, "r", encoding="utf-8") as f:
        config = json.load(f)

    v1_params[model_name] = config["hyperparameters"]

# Hàm khởi tạo mô hình với siêu tham số V1
def create_model(model_name):

    params = v1_params[model_name].copy()

    if model_name == "Random Forest":
        base_model = RandomForestClassifier(**params)

    elif model_name == "XGBoost":
        base_model = XGBClassifier(**params)

    elif model_name == "CatBoost":
        base_model = CatBoostClassifier(**params)

    else:
        raise ValueError("Mô hình không hợp lệ!")

    return MultiOutputClassifier(
        estimator=base_model,
        n_jobs=1
    )

def predict_positive_proba(model, X_val):

    probabilities = []

    for estimator in model.estimators_:

        classes = list(estimator.classes_)

        if 1 in classes:
            positive_index = classes.index(1)
            proba = estimator.predict_proba(
                X_val
            )[:, positive_index]
        else:
            proba = lib.np.zeros(len(X_val))

        probabilities.append(proba)

    return lib.np.column_stack(probabilities)

def predict_top4(probabilities):

    y_pred = lib.np.zeros_like(
        probabilities,
        dtype=int
    )

    top4_idx = lib.np.argsort(
        -probabilities,
        axis=1,
        kind="stable"
    )[:, :4]

    rows = lib.np.arange(len(probabilities))[:, None]
    y_pred[rows, top4_idx] = 1

    return y_pred

feature_sets = {}

print("Development:", df_dev.shape)
print("X gốc:", X_original.shape)
print("X Feature Engineering:", X_fe.shape)
print("Số fold:", len(fold_data))
print("PASS: Khởi tạo thành công.")

In [ ]:
# ==========================================
# FS1 - ANNUAL SCORES ONLY
# ==========================================

annual_cols = [
    col for col in score_cols
    if col.endswith("_ca_nam")
]

feature_sets["FS1_Annual"] = annual_cols

assert len(annual_cols) == 32

print("Bộ đặc trưng: FS1 - Annual Scores")
print("Số đặc trưng:", len(annual_cols))

display(X_fe[annual_cols].head())

In [ ]:
# ==========================================
# FS2 - SEMESTER SCORES ONLY
# ==========================================

semester_cols = [
    col for col in score_cols
    if col.endswith(("_hki", "_hkii"))
]

feature_sets["FS2_Semester"] = semester_cols

assert len(semester_cols) == 64

print("Bộ đặc trưng: FS2 - Semester Scores")
print("Số đặc trưng:", len(semester_cols))

display(X_fe[semester_cols].head())

In [ ]:
# ==========================================
# FS4 - ORIGINAL + TRAJECTORY FEATURES
# ==========================================

trajectory_suffixes = (
    "_mean_4years",
    "_std_4years",
    "_delta_L9_L6",
    "_change_L6_L7",
    "_change_L7_L8",
    "_change_L8_L9",
    "_trend",
    "_range_4years",
    "_mean_abs_change"
)

trajectory_cols = [
    col for col in X_fe.columns
    if col.startswith("FE_")
    and col.endswith(trajectory_suffixes)
    and not col.startswith((
        "FE_natural_",
        "FE_social_",
        "FE_technology_",
        "FE_foreign_language_"
    ))
]

fs4_cols = score_cols + trajectory_cols

feature_sets["FS4_Trajectory"] = fs4_cols

assert len(trajectory_cols) == 81
assert len(fs4_cols) == 177
assert len(fs4_cols) == len(set(fs4_cols))

print("Bộ đặc trưng: FS4 - Original + Trajectory")
print("Đặc trưng gốc:", len(score_cols))
print("Đặc trưng quỹ đạo:", len(trajectory_cols))
print("Tổng đặc trưng:", len(fs4_cols))

display(X_fe[fs4_cols].head())

In [ ]:
# ==========================================
# FEATURE IMPORTANCE ANALYSIS
# ==========================================

train_idx = fold_data[0]["train_idx"]

X_train = X_fe.iloc[train_idx][fs4_cols]
y_train = y.iloc[train_idx]

# Chỉ fit trên Training Fold 1
importance_model = create_model("Random Forest")

importance_model.fit(X_train, y_train)

# Tầm quan trọng trung bình qua 9 bộ phân loại
importance_matrix = lib.np.array([
    estimator.feature_importances_
    for estimator in importance_model.estimators_
])

importance_mean = importance_matrix.mean(axis=0)

importance_df = lib.pd.DataFrame({
    "Feature": fs4_cols,
    "Mean_Importance": importance_mean
}).sort_values(
    "Mean_Importance",
    ascending=False
).reset_index(drop=True)

print("TOP 20 FEATURES - RANDOM FOREST")
display(importance_df.head(20))

# Feature Importance theo từng nhãn
importance_per_label = lib.pd.DataFrame(
    importance_matrix.T,
    index=fs4_cols,
    columns=choice_cols
)

importance_per_label.index.name = "Feature"

# Vẽ Top 20
top20 = importance_df.head(20).iloc[::-1]

fig, ax = lib.plt.subplots(figsize=(11, 8))

ax.barh(
    top20["Feature"],
    top20["Mean_Importance"]
)

ax.set_title("Top 20 Feature Importance - Random Forest")
ax.set_xlabel("Mean Feature Importance")

lib.plt.tight_layout()

fig.savefig(
    figure_dir / "feature_importance_top20.png",
    dpi=200,
    bbox_inches="tight"
)

lib.plt.show()
lib.plt.close(fig)

print("PASS: Feature Importance chỉ được fit trên Training.")

In [ ]:
# ==========================================
# FS5 - REMOVE REDUNDANT FEATURES
# TRAINING-ONLY CORRELATION FILTER
# ==========================================

CORRELATION_THRESHOLD = 0.98

def remove_redundant_features(
    X_train,
    X_val,
    threshold=0.98
):
    """
    Xác định tương quan chỉ từ Training.
    Validation không tham gia lựa chọn đặc trưng.
    """

    # Tính ma trận tương quan trên Training
    corr_matrix = X_train.corr().abs()

    # Lấy tam giác trên của ma trận
    upper = corr_matrix.where(
        lib.np.triu(
            lib.np.ones(corr_matrix.shape, dtype=bool),
            k=1
        )
    )

    # Những cột tương quan cao với một cột đứng trước
    drop_cols = [
        col
        for col in upper.columns
        if (upper[col] >= threshold).any()
    ]

    selected_cols = [
        col for col in X_train.columns
        if col not in drop_cols
    ]

    # Áp dụng cho cả Train và Validation
    X_train_selected = X_train[selected_cols].copy()
    X_val_selected = X_val[selected_cols].copy()

    return (
        X_train_selected,
        X_val_selected,
        selected_cols,
        drop_cols
    )


# Kiểm tra thử trên Fold 1
train_idx = fold_data[0]["train_idx"]
val_idx = fold_data[0]["val_idx"]

X_train = X_fe.iloc[train_idx][fs4_cols]
X_val = X_fe.iloc[val_idx][fs4_cols]

(
    X_train_selected,
    X_val_selected,
    selected_cols,
    drop_cols
) = remove_redundant_features(
    X_train,
    X_val,
    CORRELATION_THRESHOLD
)

print("=" * 60)
print("FS5 - REDUNDANCY REMOVAL")
print("=" * 60)

print("Trước:", X_train.shape[1], "đặc trưng")
print("Sau:", X_train_selected.shape[1], "đặc trưng")
print("Số đặc trưng loại bỏ:", len(drop_cols))

assert X_train_selected.columns.equals(
    X_val_selected.columns
)

assert set(selected_cols).isdisjoint(set(drop_cols))
assert len(selected_cols) > 0

print("\nCác đặc trưng bị loại:")
display(
    lib.pd.DataFrame({
        "Removed_Feature": drop_cols
    })
)

print("\nPASS: Lựa chọn đặc trưng chỉ dựa trên Training.")

In [ ]:
# ==========================================
# KHỞI TẠO LẠI ĐẦY ĐỦ 5 BỘ ĐẶC TRƯNG
# ==========================================

feature_sets = {
    "FS1_Annual": annual_cols,
    "FS2_Semester": semester_cols,
    "FS3_Original96": score_cols,
    "FS4_Trajectory": fs4_cols
}

FEATURE_SET_NAMES = [
    "FS1_Annual",
    "FS2_Semester",
    "FS3_Original96",
    "FS4_Trajectory",
    "FS5_Redundancy"
]

# FS5 sử dụng FS4 làm đầu vào, sau đó lọc tương quan theo fold
for name in FEATURE_SET_NAMES:
    if name != "FS5_Redundancy":
        assert name in feature_sets, f"Thiếu bộ đặc trưng: {name}"
        assert all(col in X_fe.columns for col in feature_sets[name])

print("PASS: Đã khởi tạo các bộ đặc trưng.")

for name, columns in feature_sets.items():
    print(f"{name}: {len(columns)} đặc trưng")

print("FS5_Redundancy: Lựa chọn trong từng Training fold")

In [ ]:
# ==========================================
# FEATURE SET COMPARISON - 5-FOLD CV
# ==========================================

FEATURE_SET_NAMES = [
    "FS1_Annual",
    "FS2_Semester",
    "FS3_Original96",
    "FS4_Trajectory",
    "FS5_Redundancy"
]

experiment_results = []
selection_records = []
prediction_frames = []

for model_name in MODEL_NAMES:

    for feature_set_name in FEATURE_SET_NAMES:

        print("=" * 60)
        print(f"{model_name} | {feature_set_name}")
        print("=" * 60)

        for fold_info in fold_data:

            fold = fold_info["fold"]

            train_idx = fold_info["train_idx"]
            val_idx = fold_info["val_idx"]

            # FS5 bắt đầu từ FS4 rồi loại tương quan
            if feature_set_name == "FS5_Redundancy":
                columns = fs4_cols
            else:
                columns = feature_sets[feature_set_name]

            X_train = X_fe.iloc[
                train_idx
            ][columns].copy()

            X_val = X_fe.iloc[
                val_idx
            ][columns].copy()

            y_train = y.iloc[train_idx].copy()
            y_val = y.iloc[val_idx].copy()

            removed_cols = []

            # Feature selection chỉ fit trên Training fold
            if feature_set_name == "FS5_Redundancy":

                (
                    X_train,
                    X_val,
                    selected_cols,
                    removed_cols
                ) = remove_redundant_features(
                    X_train,
                    X_val,
                    CORRELATION_THRESHOLD
                )

            else:
                selected_cols = list(X_train.columns)

            # Kiểm tra không rò rỉ nhãn
            assert not (
                set(selected_cols) & set(choice_cols)
            )

            assert id_col not in selected_cols
            assert X_train.columns.equals(X_val.columns)

            # Huấn luyện mô hình mới cho từng fold
            model = create_model(model_name)

            model.fit(
                X_train,
                y_train
            )

            # Dự đoán trên Validation
            y_proba = predict_positive_proba(
                model,
                X_val
            )

            y_pred = predict_top4(y_proba)
            y_true = y_val.to_numpy(dtype=int)

            assert y_proba.shape == (len(X_val), 9)
            assert lib.np.isfinite(y_proba).all()
            assert (y_pred.sum(axis=1) == 4).all()

            # Chỉ số đánh giá
            experiment_results.append({
                "Model": model_name,
                "Feature_Set": feature_set_name,
                "Fold": fold,
                "N_Features": len(selected_cols),
                "Exact_Match_Accuracy": accuracy_score(
                    y_true, y_pred
                ),
                "Macro_F1": f1_score(
                    y_true, y_pred,
                    average="macro",
                    zero_division=0
                ),
                "Micro_F1": f1_score(
                    y_true, y_pred,
                    average="micro",
                    zero_division=0
                ),
                "Hamming_Loss": hamming_loss(
                    y_true, y_pred
                )
            })

            # Danh sách đặc trưng được chọn
            selection_records.append({
                "Model": model_name,
                "Feature_Set": feature_set_name,
                "Fold": fold,
                "N_Selected": len(selected_cols),
                "Selected_Features": json.dumps(
                    selected_cols,
                    ensure_ascii=False
                ),
                "Removed_Features": json.dumps(
                    removed_cols,
                    ensure_ascii=False
                )
            })

            # Lưu dự đoán từng học sinh
            pred_df = lib.pd.DataFrame({
                id_col: df_dev.iloc[
                    val_idx
                ][id_col].to_numpy(),
                "Model": model_name,
                "Feature_Set": feature_set_name,
                "Fold": fold
            })

            for j in range(9):
                pred_df[f"true_{j+1}"] = y_true[:, j]
                pred_df[f"pred_{j+1}"] = y_pred[:, j]
                pred_df[f"proba_{j+1}"] = y_proba[:, j]

            prediction_frames.append(pred_df)

            print(
                f"Fold {fold}: "
                f"Features={len(selected_cols)} | "
                f"Exact-match="
                f"{accuracy_score(y_true, y_pred):.4f}"
            )

print("\nPASS: Hoàn thành Feature Selection Experiment.")

In [ ]:
# ==========================================
# EVALUATION & SAVE FEATURE SELECTION
# ==========================================

fs_metrics = lib.pd.DataFrame(experiment_results)
fs_selections = lib.pd.DataFrame(selection_records)

fs_predictions = lib.pd.concat(
    prediction_frames,
    ignore_index=True
)

metric_cols = [
    "Exact_Match_Accuracy",
    "Macro_F1",
    "Micro_F1",
    "Hamming_Loss"
]

# Trung bình và độ lệch chuẩn qua 5 fold
fs_summary = (
    fs_metrics
    .groupby(["Model", "Feature_Set"])
    .agg(
        N_Features_Mean=("N_Features", "mean"),
        Exact_Match_Mean=("Exact_Match_Accuracy", "mean"),
        Exact_Match_Std=("Exact_Match_Accuracy", "std"),
        Macro_F1_Mean=("Macro_F1", "mean"),
        Macro_F1_Std=("Macro_F1", "std"),
        Micro_F1_Mean=("Micro_F1", "mean"),
        Hamming_Loss_Mean=("Hamming_Loss", "mean")
    )
    .reset_index()
)

# Đọc kết quả V1
v1_files = {
    "Random Forest": ROOT_DIR / (
        "experiments/v1_baseline/random_forest/results/"
        "rf_metrics_5fold.csv"
    ),
    "XGBoost": ROOT_DIR / (
        "experiments/v1_baseline/xgboost/results/"
        "xgb_metrics_5fold.csv"
    ),
    "CatBoost": ROOT_DIR / (
        "experiments/v1_baseline/catboost/results/"
        "cat_metrics_5fold.csv"
    )
}

v1_records = []

for model_name, path in v1_files.items():

    v1_data = lib.pd.read_csv(path)

    v1_records.append({
        "Model": model_name,
        "V1_Exact_Match": v1_data[
            "Exact_Match_Accuracy"
        ].mean(),
        "V1_Macro_F1": v1_data["Macro_F1"].mean()
    })

v1_summary = lib.pd.DataFrame(v1_records)

# So sánh từng bộ đặc trưng với V1
comparison = fs_summary.merge(
    v1_summary,
    on="Model",
    how="left",
    validate="many_to_one"
)

comparison["Delta_Exact_vs_V1"] = (
    comparison["Exact_Match_Mean"]
    - comparison["V1_Exact_Match"]
)

comparison["Delta_MacroF1_vs_V1"] = (
    comparison["Macro_F1_Mean"]
    - comparison["V1_Macro_F1"]
)

# Xếp hạng theo Exact-match Accuracy
ranking = comparison.sort_values(
    [
        "Exact_Match_Mean",
        "Macro_F1_Mean",
        "Hamming_Loss_Mean"
    ],
    ascending=[False, False, True]
)

print("=" * 60)
print("FEATURE SELECTION - SO SÁNH V1 VÀ V2")
print("=" * 60)

display(
    ranking[
        [
            "Model",
            "Feature_Set",
            "N_Features_Mean",
            "Exact_Match_Mean",
            "Exact_Match_Std",
            "Macro_F1_Mean",
            "Delta_Exact_vs_V1",
            "Delta_MacroF1_vs_V1"
        ]
    ].round(4)
)

# Vẽ biểu đồ so sánh
plot_data = fs_summary.pivot(
    index="Feature_Set",
    columns="Model",
    values="Exact_Match_Mean"
) * 100

ax = plot_data.plot(
    kind="bar",
    figsize=(12, 6)
)

ax.set_title("Feature Selection - Exact-match Accuracy")
ax.set_xlabel("Feature Set")
ax.set_ylabel("Exact-match Accuracy (%)")

lib.plt.xticks(rotation=20)
lib.plt.tight_layout()

fig = ax.get_figure()

fig.savefig(
    figure_dir / "feature_selection_comparison.png",
    dpi=200,
    bbox_inches="tight"
)

lib.plt.show()
lib.plt.close(fig)

# Lưu báo cáo
reports = {
    "feature_selection_metrics_5fold.csv": fs_metrics,
    "feature_selection_summary.csv": fs_summary,
    "feature_selection_comparison.csv": comparison,
    "feature_selection_ranking.csv": ranking,
    "feature_selection_details.csv": fs_selections,
    "feature_importance.csv": importance_df,
    "feature_importance_per_label.csv": (
        importance_per_label.reset_index()
    ),
    "feature_selection_predictions.csv": fs_predictions
}

for filename, data in reports.items():

    data.to_csv(
        table_dir / filename,
        index=False,
        encoding="utf-8-sig"
    )

# Lưu cấu hình
config = {
    "experiment": "15 - Feature Selection",
    "random_seed": RANDOM_SEED,
    "n_splits": N_SPLITS,
    "cv_source": "data/data-splits/cv_folds_v1",
    "training_source": "Development only",
    "models": MODEL_NAMES,
    "feature_sets": {
        key: columns
        for key, columns in feature_sets.items()
    },
    "FS5_base": "FS4_Trajectory",
    "FS5_method": "Training-only Pearson correlation filter",
    "correlation_threshold": CORRELATION_THRESHOLD,
    "feature_importance_method": (
        "Random Forest mean impurity importance, Fold 1 Training"
    ),
    "prediction_strategy": "Top-4 probabilities",
    "hyperparameters": v1_params,
    "independent_test_used": False
}

with open(
    config_dir / "feature_selection_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        config,
        f,
        ensure_ascii=False,
        indent=4
    )

print("\nPASS: Đã lưu kết quả Feature Selection V2.")
print("Thư mục:", output_dir.resolve())